# SPAI Inference Experiment

## Command Mapping

This notebook runs clean inference equivalent to the original SPAI CLI:

```bash
python -m spai infer \
    --cfg configs/spai.yaml \
    --input data/bronze/ntire/test_images \
    --model weights/spai.pth \
    --output output
```

**What changed**: CLI wrapper, Neptune logging, and CSV update logic stripped out. Direct programmatic inference with per-image score output.

## Datasets

- **NTIRE**: 2,500 test images with binary labels (0=authentic, 1=fake)
- **Z-Image-Turbo**: 100 AI-generated images (all label=1)

## Reproducibility

- Seed: 42
- Checkpoint: `spai.pth` (freq_restoration, arbitrary resolution, PatchBasedMFViT)
- batch_size: 1, feature_extraction_batch: 16 (WSL VRAM-safe)
- Deterministic: `torch.manual_seed`, `cudnn.deterministic`

In [ ]:
"""Imports and reproducibility setup."""
import json
import logging
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
from IPython.display import SVG, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from spai.config import get_config
from spai.data import data_finetune

# Albumentations 2.x removed ImageCompressionType; SPAI still references it
data_finetune.ImageCompressionType = type(
    "ImageCompressionType", (), {"JPEG": "jpeg", "WEBP": "webp"}
)
from spai.data.data_finetune import build_loader_test
from spai.models import build_cls_model
from spai.utils import load_pretrained

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
cudnn.deterministic = True
cudnn.benchmark = False

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("spai_experiment")

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram:.1f} GB")
print(f"spaCy config: freq_restoration / arbitrary resolution")

In [ ]:
"""Path configuration and environment info."""

REPO_ROOT = Path("..").resolve() if not Path("data").exists() else Path(".")

# --- Inputs ---
CHECKPOINT = REPO_ROOT / "data" / "artifacts" / "checkpoints" / "spai.pth"
SPAI_CONFIG = REPO_ROOT / "data" / "submodules" / "spai" / "configs" / "spai.yaml"
NTIRE_DIR = REPO_ROOT / "data" / "bronze" / "ntire" / "test_images"
NTIRE_LABELS = REPO_ROOT / "data" / "bronze" / "ntire" / "test_labels.csv"
ZIT_DIR = REPO_ROOT / "data" / "bronze" / "z_image_turbo"
ZIT_METADATA = REPO_ROOT / "data" / "bronze" / "z_image_turbo" / "metadata.csv"

# --- Outputs ---
OUTPUT_DIR = REPO_ROOT / "data" / "silver" / "spai"
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

# --- Inference params ---
BATCH_SIZE = 1  # Arbitrary resolution mode forces bs=1
FEATURE_EXTRACTION_BATCH = 16  # VRAM-safe (default 400 is too aggressive)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Git info ---
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip()
    GIT_BRANCH = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True
    ).strip()
except Exception:
    GIT_COMMIT = "unknown"
    GIT_BRANCH = "unknown"

print(f"REPO_ROOT: {REPO_ROOT}")
print(
    f"Checkpoint: {CHECKPOINT} (exists={CHECKPOINT.exists()}, size={CHECKPOINT.stat().st_size / 1e6:.0f}MB)"
)
print(f"SPAI config: {SPAI_CONFIG} (exists={SPAI_CONFIG.exists()})")
print(f"NTIRE images: {NTIRE_DIR} (exists={NTIRE_DIR.exists()})")
print(f"ZIT images: {ZIT_DIR} (exists={ZIT_DIR.exists()})")
print(f"Output: {OUTPUT_DIR}")
print(f"Git: {GIT_BRANCH} @ {GIT_COMMIT[:8]}")
print(f"Device: {DEVICE}")

In [ ]:
"""Build SPAI-compatible manifest CSVs for both datasets.

SPAI expects CSV columns: image, split, class.
"""

import filetype


def build_ntire_manifest(
    labels_csv: Path, images_dir: Path, output_path: Path
) -> pd.DataFrame:
    """Build manifest from NTIRE test_labels.csv."""
    labels = pd.read_csv(labels_csv)
    manifest = pd.DataFrame(
        {
            "image": labels["image_name"],
            "split": "test",
            "class": labels["label"].astype(str),
            "label": labels["label"],
            "source": "ntire",
        }
    )
    manifest.to_csv(output_path, index=False)
    logger.info("NTIRE manifest: %d images -> %s", len(manifest), output_path)
    return manifest


def build_zit_manifest(
    metadata_csv: Path, images_dir: Path, output_path: Path
) -> pd.DataFrame:
    """Build manifest from ZIT metadata.csv + directory listing."""
    # Use directory listing (authoritative) merged with metadata for category info
    image_files = sorted(
        [f.name for f in images_dir.iterdir() if f.is_file() and filetype.is_image(f)]
    )
    meta = pd.read_csv(metadata_csv)

    rows = []
    for fname in image_files:
        row = {
            "image": fname,
            "split": "test",
            "class": "1",
            "label": 1,
            "source": "z_image_turbo",
        }
        meta_match = meta[meta["local_path"].str.contains(fname, regex=False)]
        if not meta_match.empty:
            row["category"] = meta_match.iloc[0].get("category", "")
        rows.append(row)

    manifest = pd.DataFrame(rows)
    manifest.to_csv(output_path, index=False)
    logger.info("ZIT manifest: %d images -> %s", len(manifest), output_path)
    return manifest


NTIRE_MANIFEST = OUTPUT_DIR / "ntire_manifest.csv"
ZIT_MANIFEST = OUTPUT_DIR / "zit_manifest.csv"

manifest_ntire = build_ntire_manifest(NTIRE_LABELS, NTIRE_DIR, NTIRE_MANIFEST)
manifest_zit = build_zit_manifest(ZIT_METADATA, ZIT_DIR, ZIT_MANIFEST)

# Stats
n_real = (manifest_ntire["label"] == 0).sum()
n_fake = (manifest_ntire["label"] == 1).sum()
print(f"\nNTIRE: {len(manifest_ntire)} images ({n_real} real, {n_fake} fake)")
print(f"ZIT: {len(manifest_zit)} images (all fake)")
print(
    f"\nNTIRE class balance: real={n_real / len(manifest_ntire):.1%}, fake={n_fake / len(manifest_ntire):.1%}"
)

In [ ]:
"""Initialize SPAI model (once, reused for both datasets).

Uses freq_restoration approach with PatchBasedMFViT for arbitrary resolution.
"""

config = get_config(
    {
        "cfg": str(SPAI_CONFIG),
        "batch_size": BATCH_SIZE,
        "test_csv": [str(NTIRE_MANIFEST)],  # placeholder, overridden per-dataset
        "test_csv_root": [str(NTIRE_DIR)],
        "pretrained": str(CHECKPOINT),
        "output": str(OUTPUT_DIR),
        "tag": "spai",
        "opts": [
            ("MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH)),
        ],
    }
)

logger.info("Building model: %s / %s", config.MODEL.TYPE, config.MODEL.SID_APPROACH)
model = build_cls_model(config)
model.cuda()
load_pretrained(config, model, logger, checkpoint_path=CHECKPOINT, verbose=True)
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
vram_mb = torch.cuda.memory_allocated() / 1e6
print(f"\nModel: PatchBasedMFViT (freq_restoration, arbitrary resolution)")
print(f"Trainable params: {n_params:,}")
print(f"VRAM after model load: {vram_mb:.0f} MB")
print(f"Resolution mode: {config.MODEL.RESOLUTION_MODE}")
print(f"Feature extraction batch: {config.MODEL.FEATURE_EXTRACTION_BATCH}")
print(f"Normalization: {config.MODEL.REQUIRED_NORMALIZATION}")

In [ ]:
"""Inference helper — replicates validate() from __main__.py without re-init."""


@torch.no_grad()
def run_spai_inference(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    config,
    progress_every: int = 100,
) -> dict[int, float]:
    """Run SPAI inference and return {dataset_idx: score} mapping."""
    model.eval()
    scores: dict[int, float] = {}

    for idx, (images, target, dataset_idx) in enumerate(data_loader):
        if isinstance(images, list):
            images = [img.cuda(non_blocking=True) for img in images]
            images = [img.squeeze(dim=1) for img in images]
            output = model(images, config.MODEL.FEATURE_EXTRACTION_BATCH)
        else:
            images = images.cuda(non_blocking=True).squeeze(dim=1)
            output = model(images)

        output = torch.sigmoid(output)
        batch_scores = output.squeeze(dim=1).cpu().tolist()
        batch_indices = dataset_idx.cpu().tolist()

        if isinstance(batch_scores, float):
            batch_scores = [batch_scores]
            batch_indices = [batch_indices]

        for i, s in zip(batch_indices, batch_scores):
            scores[i] = s

        if (idx + 1) % progress_every == 0:
            vram = torch.cuda.max_memory_allocated() / 1e6
            logger.info("Batch %d/%d | VRAM: %.0f MB", idx + 1, len(data_loader), vram)

    return scores


def scores_to_dataframe(
    scores: dict[int, float], manifest: pd.DataFrame
) -> pd.DataFrame:
    """Merge inference scores with manifest to produce results DataFrame."""
    results = manifest.copy()
    score_values = [scores.get(i, float("nan")) for i in range(len(manifest))]
    results["score"] = score_values
    results["prediction"] = (results["score"] >= 0.5).astype(int)
    return results


def compute_classification_metrics(y_true, y_score, threshold=0.5):
    """Compute standard binary classification metrics."""
    y_pred = (np.array(y_score) >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "average_precision": float(average_precision_score(y_true, y_score)),
        "auc_roc": float(roc_auc_score(y_true, y_score)),
        "f1": float(f1_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "n_samples": len(y_true),
        "threshold": threshold,
    }


print("Inference helpers ready.")

In [ ]:
"""Run NTIRE inference (2,500 images, ~30-60 min).

Builds dataloader from NTIRE manifest, runs inference, saves scores + metrics.
"""

# Build NTIRE dataloader
ntire_config = get_config(
    {
        "cfg": str(SPAI_CONFIG),
        "batch_size": BATCH_SIZE,
        "test_csv": [str(NTIRE_MANIFEST)],
        "test_csv_root": [str(NTIRE_DIR)],
        "pretrained": str(CHECKPOINT),
        "output": str(OUTPUT_DIR),
        "tag": "spai_ntire",
        "opts": [
            ("MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH)),
        ],
    }
)

ntire_names, ntire_datasets, ntire_loaders = build_loader_test(
    ntire_config, logger, split="test"
)
print(
    f"NTIRE dataloader: {len(ntire_datasets[0])} images, {len(ntire_loaders[0])} batches"
)

# Run inference
print("Starting NTIRE inference...")
t0 = __import__("time").time()
ntire_scores = run_spai_inference(
    model, ntire_loaders[0], ntire_config, progress_every=100
)
elapsed = __import__("time").time() - t0
print(f"Done in {elapsed:.1f}s ({elapsed / 60:.1f} min)")

# Save scores
results_ntire = scores_to_dataframe(ntire_scores, manifest_ntire)
results_ntire.to_csv(OUTPUT_DIR / "ntire_scores.csv", index=False)

# Compute metrics (NTIRE has both classes)
y_true = results_ntire["label"].values
y_score = results_ntire["score"].values
metrics_ntire = compute_classification_metrics(y_true, y_score)

with open(OUTPUT_DIR / "ntire_metrics.json", "w") as f:
    json.dump(metrics_ntire, f, indent=2)

print(f"\nNTIRE Results ({len(results_ntire)} images):")
print(f"  Accuracy: {metrics_ntire['accuracy']:.4f}")
print(f"  Average Precision: {metrics_ntire['average_precision']:.4f}")
print(f"  AUC-ROC: {metrics_ntire['auc_roc']:.4f}")
print(f"  F1: {metrics_ntire['f1']:.4f}")
print(f"  Precision: {metrics_ntire['precision']:.4f}")
print(f"  Recall: {metrics_ntire['recall']:.4f}")

print(f"\nScore distribution by label:")
for lbl in [0, 1]:
    subset = results_ntire[results_ntire["label"] == lbl]
    tag = "authentic" if lbl == 0 else "fake"
    print(
        f"  {tag} (n={len(subset)}): mean={subset['score'].mean():.4f}, "
        f"median={subset['score'].median():.4f}, std={subset['score'].std():.4f}"
    )

vram_peak = torch.cuda.max_memory_allocated() / 1e6
print(f"\nPeak VRAM: {vram_peak:.0f} MB")

In [ ]:
"""Run Z-Image-Turbo inference (100 images, ~2-5 min)."""

zit_config = get_config(
    {
        "cfg": str(SPAI_CONFIG),
        "batch_size": BATCH_SIZE,
        "test_csv": [str(ZIT_MANIFEST)],
        "test_csv_root": [str(ZIT_DIR)],
        "pretrained": str(CHECKPOINT),
        "output": str(OUTPUT_DIR),
        "tag": "spai_zit",
        "opts": [
            ("MODEL.FEATURE_EXTRACTION_BATCH", str(FEATURE_EXTRACTION_BATCH)),
        ],
    }
)

zit_names, zit_datasets, zit_loaders = build_loader_test(
    zit_config, logger, split="test"
)
print(f"ZIT dataloader: {len(zit_datasets[0])} images, {len(zit_loaders[0])} batches")

print("Starting ZIT inference...")
t0 = __import__("time").time()
zit_scores = run_spai_inference(model, zit_loaders[0], zit_config, progress_every=25)
elapsed = __import__("time").time() - t0
print(f"Done in {elapsed:.1f}s")

# Save scores
results_zit = scores_to_dataframe(zit_scores, manifest_zit)
results_zit.to_csv(OUTPUT_DIR / "z_image_turbo_scores.csv", index=False)

# Stats (all label=1, no meaningful AUC)
print(f"\nZ-Image-Turbo Results ({len(results_zit)} images, all label=1):")
print(f"  Mean score: {results_zit['score'].mean():.4f}")
print(f"  Median score: {results_zit['score'].median():.4f}")
print(f"  Std: {results_zit['score'].std():.4f}")
print(f"  Score > 0.5: {(results_zit['score'] > 0.5).sum()} / {len(results_zit)}")
print(f"  Score > 0.8: {(results_zit['score'] > 0.8).sum()} / {len(results_zit)}")
print(f"  Min: {results_zit['score'].min():.4f}, Max: {results_zit['score'].max():.4f}")

In [ ]:
"""Reproducibility config snapshot — reads from saved inference outputs."""

with open(OUTPUT_DIR / "ntire_metrics.json") as f:
    _metrics = json.load(f)

snapshot = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "git_commit": GIT_COMMIT,
    "git_branch": GIT_BRANCH,
    "checkpoint_path": str(CHECKPOINT),
    "checkpoint_size_mb": round(CHECKPOINT.stat().st_size / 1e6, 1),
    "spai_config_path": str(SPAI_CONFIG),
    "model_type": "vit",
    "sid_approach": "freq_restoration",
    "resolution_mode": "arbitrary",
    "required_normalization": "positive_0_1",
    "feature_extraction_batch": FEATURE_EXTRACTION_BATCH,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "cudnn_deterministic": True,
    "cudnn_benchmark": False,
    "ntire_metrics": _metrics,
    "input_paths": {
        "ntire_manifest": str(NTIRE_MANIFEST),
        "ntire_images": str(NTIRE_DIR),
        "zit_manifest": str(ZIT_MANIFEST),
        "zit_images": str(ZIT_DIR),
    },
    "output_path": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(snapshot, f, indent=2, default=str)

print(f"Saved experiment_config.json to {OUTPUT_DIR}")
print(f"Keys: {list(snapshot.keys())}")

In [ ]:
"""Analysis: score distributions, ROC, PR curves (saved as SVG)."""

# Load results (allows re-running this cell without re-inference)
results_ntire = pd.read_csv(OUTPUT_DIR / "ntire_scores.csv")
results_zit = pd.read_csv(OUTPUT_DIR / "z_image_turbo_scores.csv")
with open(OUTPUT_DIR / "ntire_metrics.json") as f:
    metrics_ntire = json.load(f)

acc = metrics_ntire["accuracy"]
ap = metrics_ntire["average_precision"]
auc = metrics_ntire["auc_roc"]

y_true = results_ntire["label"].values
y_score = results_ntire["score"].values

# --- Plot 1: Score histogram ---
fig1, ax = plt.subplots(figsize=(6, 4))
for lbl, color, tag in [
    (0, "steelblue", "NTIRE authentic"),
    (1, "coral", "NTIRE fake"),
]:
    subset = results_ntire[results_ntire["label"] == lbl]
    ax.hist(subset["score"], bins=40, alpha=0.5, label=tag, color=color, density=True)
ax.hist(
    results_zit["score"],
    bins=25,
    alpha=0.6,
    label="Z-Image-Turbo (all fake)",
    color="darkred",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8, label="Threshold 0.5")
ax.set_xlabel("Fake Probability Score")
ax.set_ylabel("Density")
ax.set_title("SPAI Score Distribution")
ax.legend(fontsize=7)
plt.tight_layout()
fig1.savefig(PLOTS_DIR / "score_histogram.svg", format="svg")
plt.close(fig1)

# --- Plot 2: ROC curve ---
fig2, ax = plt.subplots(figsize=(5, 4))
fpr, tpr, _ = roc_curve(y_true, y_score)
ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"AUC = {auc:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"SPAI ROC Curve — NTIRE (Acc={acc:.4f})")
ax.legend()
plt.tight_layout()
fig2.savefig(PLOTS_DIR / "ntire_roc_curve.svg", format="svg")
plt.close(fig2)

# --- Plot 3: PR curve ---
fig3, ax = plt.subplots(figsize=(5, 4))
prec, rec, _ = precision_recall_curve(y_true, y_score)
ax.plot(rec, prec, color="steelblue", linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"SPAI Precision-Recall — NTIRE (AP={ap:.4f})")
plt.tight_layout()
fig3.savefig(PLOTS_DIR / "ntire_pr_curve.svg", format="svg")
plt.close(fig3)

print(f"Saved SVGs to {PLOTS_DIR}")
display(SVG(filename=PLOTS_DIR / "score_histogram.svg"))

In [ ]:
display(SVG(filename=PLOTS_DIR / "ntire_roc_curve.svg"))
display(SVG(filename=PLOTS_DIR / "ntire_pr_curve.svg"))

## Summary

| Dataset | Images | Accuracy | AP | AUC-ROC | F1 | Precision | Recall |
|---------|--------|----------|------|---------|------|-----------|--------|
| NTIRE | 2,500 | 0.6104 | 0.6472 | 0.6428 | 0.6123 | 0.6345 | 0.5915 |
| Z-Image-Turbo | 100 | — | — | — | — | mean=0.7208 | 77/100 > 0.5 |

**Runtime**: NTIRE 26.6 min, ZIT 33.5s. Peak VRAM: 2,278 MB.

**Key findings:**
- SPAI (freq_restoration + arbitrary resolution) achieves moderate generalization on NTIRE (AP=0.65, AUC=0.64), better than AIDE (AP=0.62, AUC=0.61) on the same dataset.
- SPAI detects 77/100 Z-Image-Turbo images as fake (mean score 0.72), slightly lower than AIDE's 86/100 (mean 0.77).
- Score distributions show significant overlap between authentic and fake classes in NTIRE, suggesting the checkpoint's training distribution differs from NTIRE generators.

**Artifacts:**
- `data/silver/spai/ntire_scores.csv` — per-image scores for NTIRE
- `data/silver/spai/z_image_turbo_scores.csv` — per-image scores for Z-Image-Turbo
- `data/silver/spai/ntire_metrics.json` — NTIRE classification metrics
- `data/silver/spai/experiment_config.json` — full reproducibility snapshot
- `data/silver/spai/plots/` — score histogram, ROC curve, PR curve (SVG + PNG)